# NLBSE'27 Multitask Code Classification — Final Model

**Model**: DistilRoBERTa-base (6 layers, 82M params) with two-output classification head  
**Baseline**: CodeBERT-base (12 layers, 125M params) — competition score 0.71  

Key improvements over baseline:
- ~50% fewer FLOPs from smaller model (large efficiency score gain)
- Focal loss with class-weighted BCE for imbalanced MAT class
- Per-task threshold tuning to independently maximise F1 on each head

See `DECISIONS.md` for rationale behind every choice.

## Setup

In [ ]:
!pip install datasets transformers torch accelerate -q

In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import time
import pandas as pd
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW
from tqdm import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CONFIG = {
    'model_name':       'distilroberta-base',
    'max_length':       512,
    'batch_size_train': 32,
    'batch_size_eval':  512,
    'num_epochs':       3,
    'learning_rate':    2e-5,
    'weight_decay':     0.01,
    'warmup_ratio':     0.1,
    'dropout':          0.1,
    'focal_gamma':      2.0,
    'val_split':        0.05,
    'seed':             42,
    'save_path':        'best_model.pt',
}
print('Config loaded.')

## 1. Load & Explore Data

In [ ]:
ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
train_df = ds['train'].to_pandas()
test_df  = ds['test'].to_pandas()
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')
print(train_df.head(2))

In [ ]:
train_labels = np.array(train_df['Label'].tolist())
test_labels  = np.array(test_df['Label'].tolist())

label_names = ['Vulnerability', 'MAT']
print('Train class distribution:')
for i, name in enumerate(label_names):
    pos = train_labels[:, i].sum()
    pct = 100 * pos / len(train_labels)
    print(f'  {name}: {pos:,} positive ({pct:.1f}%)')

pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(device)
print(f'\nPos weights: Vulnerability={pos_weight[0]:.2f}, MAT={pos_weight[1]:.2f}')

## 2. Data Preparation

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labs, val_labs = train_test_split(
    train_df['Function'].tolist(),
    train_labels,
    test_size=CONFIG['val_split'],
    random_state=CONFIG['seed'],
    stratify=[
        str(r[0]) + str(r[1]) for r in train_labels
    ]
)
print(f'Train: {len(train_texts):,}  |  Val: {len(val_texts):,}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class CodeDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = torch.tensor(np.array(labels), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def make_collate_fn(tok, max_len):
    def collate_fn(batch):
        texts, labels = zip(*batch)
        enc = tok(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        return enc['input_ids'], enc['attention_mask'], torch.stack(list(labels))
    return collate_fn


collate_fn = make_collate_fn(tokenizer, CONFIG['max_length'])

train_dataset = CodeDataset(train_texts, train_labs)
val_dataset   = CodeDataset(val_texts,   val_labs)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size_train'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size_eval'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 3. Model Architecture

In [ ]:
class MultitaskClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(self.dropout(cls))


model = MultitaskClassifier(
    CONFIG['model_name'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M  |  Trainable: {trainable/1e6:.1f}M')

## 4. Loss Function — Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """
    Binary focal loss for multi-label classification.
    Downweights easy examples so the model focuses on hard, misclassified ones.
    pos_weight amplifies the minority class gradient (same semantics as BCE pos_weight).
    """
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('pos_weight', pos_weight)

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction='none'
        )
        pt = torch.exp(-bce)                      # probability of correct class
        focal_weight = (1.0 - pt) ** self.gamma   # down-weight easy examples
        return (focal_weight * bce).mean()


criterion = FocalLoss(gamma=CONFIG['focal_gamma'], pos_weight=pos_weight)
print('Focal loss initialised with pos_weight:', pos_weight.cpu().numpy())

## 5. Training

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in tqdm(loader, desc='  train', leave=False):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def eval_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)
        logits         = model(input_ids, attention_mask)
        total_loss    += criterion(logits, labels).item()
    return total_loss / len(loader)


print('Training functions defined.')

In [ ]:
total_steps   = len(train_loader) * CONFIG['num_epochs']
warmup_steps  = int(total_steps * CONFIG['warmup_ratio'])

optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float('inf')

print(f'Total steps: {total_steps:,}  |  Warmup steps: {warmup_steps:,}')
print('Starting training...\n')

for epoch in range(1, CONFIG['num_epochs'] + 1):
    t0       = time.time()
    tr_loss  = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss = eval_loss(model, val_loader, criterion, device)
    elapsed  = time.time() - t0

    print(f'Epoch {epoch}/{CONFIG["num_epochs"]}  '
          f'train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  ({elapsed/60:.1f} min)')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'val_loss':         val_loss,
            'config':           CONFIG,
        }, CONFIG['save_path'])
        print(f'  -> Best model saved (val_loss={val_loss:.4f})')

print('\nTraining complete.')

## 6. Per-Task Threshold Optimisation

Default threshold 0.5 is suboptimal for imbalanced classes.  
We sweep thresholds on the **validation split** (not test) to avoid leakage.

In [ ]:
checkpoint = torch.load(CONFIG['save_path'], map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]} (val_loss={checkpoint["val_loss"]:.4f})')

In [ ]:
@torch.no_grad()
def get_probs(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for input_ids, attention_mask, labels in tqdm(loader, desc='Collecting probs'):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        logits         = model(input_ids, attention_mask)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)


def find_best_threshold(probs, labels, task_idx):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs[:, task_idx] > t).astype(int)
        tp    = int(((preds == 1) & (labels[:, task_idx] == 1)).sum())
        fp    = int(((preds == 1) & (labels[:, task_idx] == 0)).sum())
        fn    = int(((preds == 0) & (labels[:, task_idx] == 1)).sum())
        p     = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r     = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1    = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


val_probs, val_labels_arr = get_probs(model, val_loader, device)

thresholds = []
for i, name in enumerate(label_names):
    t, f1 = find_best_threshold(val_probs, val_labels_arr, i)
    thresholds.append(t)
    print(f'{name}: best threshold = {t:.2f}  (val F1 = {f1:.4f})')

torch.save({
    'epoch':            checkpoint['epoch'],
    'model_state_dict': model.state_dict(),
    'val_loss':         checkpoint['val_loss'],
    'thresholds':       thresholds,
    'config':           CONFIG,
}, CONFIG['save_path'])
print('\nThresholds saved to checkpoint.')

## 7. Competition Evaluation

Evaluation format matches the baseline notebook exactly:  
FLOPs and runtime measured with `torch.profiler` and each batch run twice (divided by 2).

In [ ]:
total_flops = 0
total_time  = 0
predictions = []

test_data = test_df

test_encodings = tokenizer(
    test_data['Function'].tolist(),
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=CONFIG['max_length']
)

eval_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask']
)
eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size_eval'])

model.eval()
pbar = tqdm(total=len(test_data), desc='Predicting...')

for batch in eval_loader:
    input_ids, attention_mask = batch
    input_ids      = input_ids.to(device, non_blocking=True)
    attention_mask = attention_mask.to(device, non_blocking=True)

    begin = time.time()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            with torch.profiler.profile(with_flops=True) as p:
                for _ in range(2):
                    logits = model(input_ids, attention_mask)
    total_time  += time.time() - begin
    total_flops += sum(k.flops for k in p.key_averages()) / 1e9

    probs = torch.sigmoid(logits).cpu().numpy()
    for row in probs:
        predictions.append([
            int(row[0] > thresholds[0]),
            int(row[1] > thresholds[1])
        ])

    del input_ids, attention_mask, logits, probs
    torch.cuda.empty_cache()
    pbar.update(len(predictions) - pbar.n)

pbar.close()
print('Inference complete.')

In [ ]:
predictions = np.array(predictions)
labels_arr  = np.array(test_data['Label'].tolist())
scores      = []

print('----------------RESULTS--------------------')
print(f'Compute in GFLOPs:       {total_flops / 2:.2f}')
print(f'Avg runtime in seconds:  {total_time  / 2:.4f}')
print()

class_metrics = {}
for class_idx, name in enumerate(label_names):
    tp = int(((predictions[:, class_idx] == 1) & (labels_arr[:, class_idx] == 1)).sum())
    fp = int(((predictions[:, class_idx] == 1) & (labels_arr[:, class_idx] == 0)).sum())
    tn = int(((predictions[:, class_idx] == 0) & (labels_arr[:, class_idx] == 0)).sum())
    fn = int(((predictions[:, class_idx] == 0) & (labels_arr[:, class_idx] == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (tp + tn) / len(labels_arr)

    class_metrics[name] = dict(tp=tp, fp=fp, tn=tn, fn=fn,
                                precision=precision, recall=recall, f1=f1, accuracy=accuracy)
    scores.append({'cat': name, 'precision': precision, 'recall': recall, 'f1': f1})

scores_df = pd.DataFrame(scores)
print(scores_df.to_string(index=False))

In [ ]:
max_avg_runtime = 1000
max_avg_flops   = 3_000_000

avg_f1      = scores_df['f1'].mean()
avg_runtime = total_time  / 2
avg_flops   = total_flops / 2

def competition_score(avg_f1, avg_runtime, avg_flops):
    return (
        0.6 * avg_f1
        + 0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0)
        + 0.2 * max((max_avg_flops   - avg_flops)   / max_avg_flops,   0)
    )

final_score = competition_score(avg_f1, avg_runtime, avg_flops)

print(f'Avg F1:       {avg_f1:.4f}')
print(f'Avg runtime:  {avg_runtime:.2f}s  (max 1000s)')
print(f'Avg GFLOPs:   {avg_flops:,.0f}  (max 3,000,000)')
print()
print(f'F1 component:      {0.6 * avg_f1:.4f}')
print(f'Runtime component: {0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0):.4f}')
print(f'FLOPs component:   {0.2 * max((max_avg_flops - avg_flops) / max_avg_flops, 0):.4f}')
print()
print(f'FINAL SCORE: {final_score:.4f}  (baseline: 0.71)')